# Datamine ALPHCODE Process Example

This notebook demonstrates how to configure and run the **`alphcode`** process wrapper in `dmstudio`.

## Process Description

## ALPHCODE

Converts and back-converts an alpha numeric code to a numeric code.

IN must contain either:

* an alphanumeric field to code with numeric values (optionally based on a dictionary), or;

* a numeric code to be back-converted to an alpha equivalent based on a dictionary.

The conversion method is determined by the @**INVERSE** parameter.

When **INVERSE** =0

* You can code from alpha to numeric without an input dictionary (**INDICT**), and output dictionary (**OUTDICT**) is optional.

* You can code from alpha to numeric using an input dictionary.

## WHEN **INVERSE** =1

* You can code from numeric to alpha with an input dictionary.

* If no input dictionary is provided, the process will fail.

### Dictionary Files

Dictionary file data can be specified (optionally) to decode either input (**INDICT**) and output (**OUTDICT**) data, or both.

The input dictionary file contains equivalent alphanumeric and numeric value-pairs and should have the same format as an **OUTDICT** file, which is created when converting alpha to numeric values. * **FIELD** and * **NFIELD** must be present. This file may be used when @**INVERSE** =0 or @**INVERSE** =1. If this file is not present when @**INVERSE** =0, numeric coding is created and written to **OUTDICT** (if provided). This file is _required_ when @**INVERSE** =1. If **INDICT** and **OUTDICT** are both specified, **OUTDICT** is a copy of **INDICT**.

The output dictionary file is created when generating alpha values from numeric values (**INVERSE** =0). **OUTDICT** provides a record of the numeric values created that correspond to the input alpha values. It also contains a count of the number of records with each alpha value. If **INDICT** and **OUTDICT** are specified, **OUTDICT** is a copy of **INDICT**. The output dictionary is always optional.

### Input Files:

* **in** (Table):
  File containing alpha field to code with numeric values (optionally based on a
  dictionary); or a numeric code to be back-converted to an alpha equivilent based on a
  dictionary.
  Required=Yes

* **indict** (Table):
  Input dictionary file. See "Dictionary Files".
  Required=No

### Output Files:

* **out** (Table):
  Output file containing fields in the input file plus a numeric field containing the
  alpha coded values. If file is back-converted, this contains the alpha equivilent for
  the numeric field.
  Required=Yes

* **outdict** (Table):
  Output dictionary file. See "Dictionary Files"
  Required=No

### Fields:

* **field** (Any : IN):
  The alpha field to use to create corresponding numeric values. This field should either
  exist in IN or INDICT
  Default=Undefined
  Required=Yes

* **nfield** (Any : IN):
  The numeric field containing the numeric values corresponding the alpha values in
  *FIELD. This should be a new field name or exist in INDICT
  Default=Undefined
  Required=Yes

### Parameters:

* **inverse**:
  A flag to determine the type of conversion to perform: =0 : Convert input FIELD in IN
  file from Alpha to Numeric, using INDICT if supplied. =1 : Back-convert input FIELD in
  IN file from Numeric to Alpha using INDICT.
  Range=0,1
  Values=0,1
  Default=0
  Required=Yes

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('alphcode')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute alphcode
print("Running alphcode...")
dm_cmd.alphcode(
    in_i='t_assays',  # required
    indict_i='optional',  # required
    out_o='t_alphcode_out',  # required
    outdict_o='t_alphcode_out',  # required
    field_f='optional',  # required
    nfield_f='optional',  # required
    inverse_p='required_val',  # required
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("alphcode execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_alphcode_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")